# Random Forests

Decision trees have a lot of problems. Most notably, they are often not very accurate and overfit to the training data. Random forests are a way to fix that problem. It works by building many decision trees instead of just one, and using the combined prediction of all the trees (like a forest) to decide on the final predicted output.

## Constructing a Random Forest

### Bootstrapped Dataset
The first step is to create a bootstrapped dataset. The boostrapped dataset is a randomply sampled subcollection of the main training set, but it is not exactly a set, because we can repeat examples. Doing this procedure usually leaves out around 1/3 of the training set.



### Constructing the First Tree

Now, we can start constructing our first decision tree, beginning with the first split. In a decision tree, we look at all possible features and possible splits and then pick the one that results in the most pure split. However, in a random forest method, we do not look at all possible features. Instead, we randomly select a subset (the number we decide or optimize for using cross-validation) to decide splits on. So for example, if we have 4 input features, then we might only consider the splits for two of them and make a best choice out of those two. 

Once we've made the first split, we continue building out the tree and picking out splits. Each time, we again pick a random subset of the features and decide a split based on those variables. Note that the same feature can be selected multiple times because of the randomness, so the same feature can be used multiple times in a decision tree. Once we've hit some constraints similar to a regular decision tree, we've constructed a decision tree that is more random than deterministic, because at each step we randomly sampled features as candidate splits.


### The Full Forest

Random Forest repeats the technique to construct the first tree many times to construct many slightly random different trees (hence the name random forest). 

### Making a Prediction

When we want to make a prediction for a new input, we pass the data through all the different trees in the random forest. Each tree will give a slightly different output, or there will be a mix of different categorical outputs. Then, we either take the output that appeared the most times in the trees (for classification), or we take some average/median of the outputs (for regression).

### Scoring a Random Forest

We can determine how good a random forest is by feeding in the training examples that did not get randomly selected into the bootstrapped dataset. Then, we measure how accurate our random forest is by the proportion of Out-Of-Bag (out of bootstrapped dataset) samples that were correctly classified by the Random Forest. The proportion that was incorrectly labeled is called the __Out-Of-Bag error__.

We can compare the Out-Of-Bag error when we are picking a different amount of variables per step in order to pick the optimal amount.

We typically start with the __square root__ of the number of features, and then try a few settings above and below that value.

## Filling in Missing Data with Random Forests

### Filling in missing values in the training set

To fill in missing values in the training set, we can start by making a bad guess for each. For example, we can take the most common classification of all the other examples, or we can take the median of the others if it is numerical.

Then, with this filled in data, we can build a random forest as usual. Afterwards, we can run each example down the forest, starting with the first tree. We store a proximity matrix that is $n \times n$, where $n$ is the number of examples. Each time two examples $i$ and $j$ end up at the same leaf in one of the decision trees in the forest, we increment the element at $[i][j]$ (note that by symmetry, we also increment $[i][j]$). After performing this for all trees, we have a count of every time a sample $i$ and $j$ ended up at the same leaf in a tree (we can ignore $[i][i]$ because it is pointless to compare similarity of the same two examples).

Then, we can divide each element of the proximity by the total number of trees. 

Now that we have a full proximity matrix, we can make better filled-in missing values.

- If an example is missing a categorical feature, then we compare the average of the proximities for all the examples of each category, and we select whichever category has the greatest average proximity.

- For a numerical feature, you can use the proximities to calculate a weighted average of all the other examples' numerical values.

Now, we have better guesses for each of the missing values. __We can repeat this entire procedure multiple times (on average 6 or 7 times) until the missing values converge__.

Two notes:
- Proximities to an example $i$ do not have to add up to 1.
- We typically perform the updated imputation (the weighted average or the average of the proximities) only on the examples where we have the actual values of the feature.

By doing $1 - proximity$, you can also compute distance and construct a heat map.

### Missing Data in a new sample that you want to categorize

First, we create two copies of the test sample data. One, where the output classification is yes, and one where it is no (or the arbitrary number of classifications). 

Then, we use the iterative method mentioned before to make a good guess about the missing values. Once we have good guesses about the missing features, we can rerun the two samples down the trees in the forest, and we see which one is correctly labeled by the forest the most times.

We pick the one that was correctly labeled more often. By doing this, we both fill in the missing data and also automatically get the output from the decision tree.